# Reproducible PTB-XL MI Prediction Notebook

This notebook is designed to match the paper:

**A Lightweight Agentic Framework for Interpretable ECG-Based Myocardial Infarction Prediction on PTB-XL: Bridging the Translational Gap**

It includes:

1. Robust PTB-XL download/loading for Google Colab
2. SCP-to-MI label mapping
3. Official PTB-XL split: folds 1–8 train, fold 9 validation, fold 10 test
4. Train-only Z-score normalization
5. Lightweight 1D CNN
6. `BCEWithLogitsLoss` training
7. Best-validation-checkpoint selection
8. Fold-10 AUROC/AUPRC evaluation
9. ROC/PR curve generation
10. Deterministic risk-summary workflow

**Important:** After running the notebook, update the paper with the exact AUROC/AUPRC values printed by this notebook.

## 1. Install dependencies

In [ ]:
# In Colab this installs the required packages.
# If you are running locally and already have these packages, you can skip this cell.
!pip -q install wfdb scikit-learn pandas numpy matplotlib torch torchvision tqdm requests

## 2. Imports and reproducibility settings

In [ ]:
import os
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
)

import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic settings improve reproducibility but can slow training.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 3. Configure project directory

In [ ]:
# This cell is safe for Colab and local execution.
# In Colab, Google Drive is strongly recommended because PTB-XL is large.

USE_GOOGLE_DRIVE = True  # Set to False if you want to use local /content storage in Colab.

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ptbxl_mi_project")
elif IN_COLAB:
    PROJECT_DIR = Path("/content/ptbxl_mi_project")
else:
    PROJECT_DIR = Path("./ptbxl_mi_project")

DATA_DIR = PROJECT_DIR / "ptbxl"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = DATA_DIR / "ptbxl_database.csv"
scp_path = DATA_DIR / "scp_statements.csv"

print("IN_COLAB:", IN_COLAB)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("metadata_path:", metadata_path)
print("scp_path:", scp_path)

## 4. Download PTB-XL metadata and 100 Hz records

In [ ]:
# This cell avoids wfdb.dl_database(), which can fail when PhysioNet gives temporary 502 errors.
# It downloads only the 100 Hz ECG files used in the paper: records100/.
#
# If the download stops or Colab disconnects, run this same cell again.
# Because wget uses -c and -N, it will resume/skip already downloaded files.

BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3"

print("Downloading metadata files...")
!wget -c --tries=50 --waitretry=20 --retry-connrefused --read-timeout=60 --timeout=30 \
  -O "{metadata_path}" "{BASE_URL}/ptbxl_database.csv"

!wget -c --tries=50 --waitretry=20 --retry-connrefused --read-timeout=60 --timeout=30 \
  -O "{scp_path}" "{BASE_URL}/scp_statements.csv"

print("Downloading/resuming records100 directory...")
!wget -r -N -c -np -nH --cut-dirs=3 \
  --tries=50 \
  --waitretry=20 \
  --retry-connrefused \
  --read-timeout=60 \
  --timeout=30 \
  --no-check-certificate \
  -R "index.html*" \
  -P "{DATA_DIR}" \
  "{BASE_URL}/records100/"

print("Metadata exists:", metadata_path.exists(), metadata_path)
print("SCP exists:", scp_path.exists(), scp_path)

assert metadata_path.exists(), f"Missing {metadata_path}"
assert scp_path.exists(), f"Missing {scp_path}"

print("Download/check cell finished.")

## 5. Load metadata and create binary MI labels

In [ ]:
df = pd.read_csv(metadata_path, index_col="ecg_id")
scp_statements = pd.read_csv(scp_path, index_col=0)

print("Metadata shape:", df.shape)
print("SCP statements shape:", scp_statements.shape)
print(df[["filename_lr", "strat_fold", "scp_codes"]].head())
print(scp_statements.head())

In [ ]:
# PTB-XL stores SCP codes as strings like "{'NORM': 100.0, 'IMI': 15.0}".
df["scp_codes_dict"] = df["scp_codes"].apply(ast.literal_eval)

# Keep diagnostic SCP codes only.
diagnostic_scp = scp_statements[scp_statements["diagnostic"] == 1.0]

def aggregate_diagnostic_classes(scp_code_dict):
    diagnostic_classes = []
    for code in scp_code_dict.keys():
        if code in diagnostic_scp.index:
            diagnostic_classes.append(diagnostic_scp.loc[code, "diagnostic_class"])
    return list(set(diagnostic_classes))

df["diagnostic_superclass"] = df["scp_codes_dict"].apply(aggregate_diagnostic_classes)
df["MI_label"] = df["diagnostic_superclass"].apply(lambda classes: int("MI" in classes))

print(df[["scp_codes", "diagnostic_superclass", "MI_label", "strat_fold"]].head())
print("\nClass distribution:")
print(df["MI_label"].value_counts())
print("\nClass distribution normalized:")
print(df["MI_label"].value_counts(normalize=True))

## 6. Verify that ECG files exist

In [ ]:
# Each filename_lr points to records100/... without extension.
# We require both .hea and .dat files.

def find_missing_record_files(metadata_df, data_dir, max_report=20):
    missing = []
    for fname in tqdm(metadata_df["filename_lr"].values, desc="Checking ECG files"):
        record_base = data_dir / fname
        hea_path = Path(str(record_base) + ".hea")
        dat_path = Path(str(record_base) + ".dat")
        if not hea_path.exists():
            missing.append(str(hea_path))
        if not dat_path.exists():
            missing.append(str(dat_path))
        if len(missing) >= max_report:
            # Continue would show only first files; break is faster for quick diagnosis.
            break
    return missing

missing = find_missing_record_files(df, DATA_DIR)

if len(missing) > 0:
    print("Some ECG files are missing. First missing files:")
    for m in missing:
        print(m)
    raise FileNotFoundError(
        "Some records100 files are missing. Re-run the download cell above. "
        "The wget command is resumable and should continue from where it stopped."
    )
else:
    print("Initial file check passed. Required records100 files appear to exist.")

## 7. Official PTB-XL fold split

In [ ]:
# Official split used in the paper:
# folds 1-8: training
# fold 9: validation
# fold 10: final test

train_df = df[df["strat_fold"].isin([1, 2, 3, 4, 5, 6, 7, 8])].copy()
val_df = df[df["strat_fold"] == 9].copy()
test_df = df[df["strat_fold"] == 10].copy()

print("Train:", train_df.shape, train_df["MI_label"].value_counts().to_dict())
print("Val:", val_df.shape, val_df["MI_label"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["MI_label"].value_counts().to_dict())

assert set(train_df["strat_fold"].unique()).issubset(set(range(1, 9)))
assert set(val_df["strat_fold"].unique()) == {9}
assert set(test_df["strat_fold"].unique()) == {10}

## 8. Load 100 Hz ECG signals

In [ ]:
def load_raw_data(metadata_subset, data_dir):
    '''
    Loads PTB-XL 100 Hz ECG records.

    wfdb returns signal shape [1000, 12].
    For PyTorch Conv1D, we transpose to [12, 1000].
    '''
    signals = []
    labels = []

    for _, row in tqdm(metadata_subset.iterrows(), total=len(metadata_subset), desc="Loading ECG records"):
        record_path = data_dir / row["filename_lr"]

        try:
            signal, meta = wfdb.rdsamp(str(record_path))
        except Exception as e:
            raise RuntimeError(f"Failed to read {record_path}. Error: {e}")

        # Expected raw shape: [time, leads] = [1000, 12]
        if signal.shape != (1000, 12):
            raise ValueError(f"Unexpected signal shape {signal.shape} for {record_path}")

        # Convert to [leads, time] = [12, 1000]
        signal = signal.T.astype(np.float32)

        signals.append(signal)
        labels.append(row["MI_label"])

    X = np.stack(signals).astype(np.float32)
    y = np.array(labels).astype(np.float32)

    return X, y

X_train, y_train = load_raw_data(train_df, DATA_DIR)
X_val, y_val = load_raw_data(val_df, DATA_DIR)
X_test, y_test = load_raw_data(test_df, DATA_DIR)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

assert X_train.shape[1:] == (12, 1000), "Expected input shape [N, 12, 1000]"
assert X_val.shape[1:] == (12, 1000), "Expected input shape [N, 12, 1000]"
assert X_test.shape[1:] == (12, 1000), "Expected input shape [N, 12, 1000]"

## 9. Train-only Z-score normalization

In [ ]:
# Fit normalization statistics only on training data to avoid data leakage.
# Per-lead mean/std: shape [1, 12, 1]

train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Train mean after normalization:", X_train.mean())
print("Train std after normalization:", X_train.std())

np.save(OUTPUT_DIR / "train_mean.npy", train_mean)
np.save(OUTPUT_DIR / "train_std.npy", train_std)

## 10. PyTorch Dataset and DataLoaders

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_loader = DataLoader(
    ECGDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2 if IN_COLAB else 0,
    pin_memory=True if device.type == "cuda" else False,
)

val_loader = DataLoader(
    ECGDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2 if IN_COLAB else 0,
    pin_memory=True if device.type == "cuda" else False,
)

test_loader = DataLoader(
    ECGDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2 if IN_COLAB else 0,
    pin_memory=True if device.type == "cuda" else False,
)

xb, yb = next(iter(train_loader))
print("Batch X:", xb.shape)
print("Batch y:", yb.shape)

## 11. Lightweight 1D CNN matching the paper

In [ ]:
class LightweightECGNet(nn.Module):
    def __init__(self, in_channels=12):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        # x shape: [B, 12, 1000]
        x = self.features(x)
        x = self.global_pool(x).squeeze(-1)  # [B, 128]
        x = self.dropout(x)
        logit = self.classifier(x).view(-1)  # [B]
        return logit

model = LightweightECGNet().to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

with torch.no_grad():
    dummy = torch.randn(4, 12, 1000).to(device)
    out = model(dummy)
    print("Dummy output shape:", out.shape)

## 12. Train model and save best validation checkpoint

In [ ]:
criterion = nn.BCEWithLogitsLoss()

# Matches paper setting: Adam with lr=1e-3.
# Weight decay is included here for reproducibility; keep the paper wording aligned with this.
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 5
best_val_loss = float("inf")
best_model_path = OUTPUT_DIR / "best_model.pth"

history = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - train"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.float().to(device).view(-1)

        optimizer.zero_grad()
        logits = model(X_batch).view(-1)

        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss = train_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - val"):
            X_batch = X_batch.to(device)
            y_batch = y_batch.float().to(device).view(-1)

            logits = model(X_batch).view(-1)
            loss = criterion(logits, y_batch)

            val_loss += loss.item() * X_batch.size(0)

    val_loss = val_loss / len(val_loader.dataset)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
    }
    history.append(row)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best checkpoint saved to {best_model_path}")

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
history_df

## 13. Plot training history

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_validation_loss.png", dpi=300)
plt.show()

## 14. Final evaluation on Fold 10 test set

In [ ]:
def collect_probabilities(model, loader, device):
    model.eval()
    all_probs = []
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in tqdm(loader, desc="Collecting test probabilities"):
            X_batch = X_batch.to(device)
            logits = model(X_batch).view(-1)
            probs = torch.sigmoid(logits)

            all_logits.extend(logits.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return (
        np.array(all_logits),
        np.array(all_probs),
        np.array(all_labels).astype(int),
    )

# Load the best validation checkpoint before test evaluation
model.load_state_dict(torch.load(best_model_path, map_location=device))

test_logits, test_probs, test_labels = collect_probabilities(model, test_loader, device)

test_auroc = roc_auc_score(test_labels, test_probs)
test_auprc = average_precision_score(test_labels, test_probs)

print(f"Test AUROC: {test_auroc:.4f}")
print(f"Test AUPRC: {test_auprc:.4f}")

results = {
    "test_auroc": float(test_auroc),
    "test_auprc": float(test_auprc),
    "n_test": int(len(test_labels)),
    "n_positive": int(test_labels.sum()),
    "n_negative": int(len(test_labels) - test_labels.sum()),
}

pd.DataFrame([results]).to_csv(OUTPUT_DIR / "test_results.csv", index=False)
pd.DataFrame([results])

## 15. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(test_labels, test_probs)
precision, recall, _ = precision_recall_curve(test_labels, test_probs)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUROC = {test_auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on PTB-XL Fold 10")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=300)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f"AUPRC = {test_auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve on PTB-XL Fold 10")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "precision_recall_curve.png", dpi=300)
plt.show()

## 16. Optional threshold-dependent metrics at 0.5

In [ ]:
threshold = 0.5
test_pred = (test_probs >= threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_pred))

print("\nClassification Report:")
print(classification_report(test_labels, test_pred, digits=4))

## 17. Optional bootstrap confidence intervals

In [ ]:
def bootstrap_metric_ci(y_true, y_score, metric_fn, n_bootstraps=1000, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []

    for _ in tqdm(range(n_bootstraps), desc=f"Bootstrapping {metric_fn.__name__}"):
        indices = rng.integers(0, n, n)
        y_true_sample = y_true[indices]
        y_score_sample = y_score[indices]

        # Skip samples with only one class
        if len(np.unique(y_true_sample)) < 2:
            continue

        scores.append(metric_fn(y_true_sample, y_score_sample))

    scores = np.array(scores)
    lower = np.percentile(scores, 100 * alpha / 2)
    upper = np.percentile(scores, 100 * (1 - alpha / 2))
    mean = np.mean(scores)

    return mean, lower, upper

# Reduce n_bootstraps to 200 if Colab is slow.
N_BOOTSTRAPS = 1000

auroc_mean, auroc_low, auroc_high = bootstrap_metric_ci(
    test_labels, test_probs, roc_auc_score, n_bootstraps=N_BOOTSTRAPS, seed=SEED
)

auprc_mean, auprc_low, auprc_high = bootstrap_metric_ci(
    test_labels, test_probs, average_precision_score, n_bootstraps=N_BOOTSTRAPS, seed=SEED
)

print(f"AUROC mean: {auroc_mean:.4f}, 95% CI: [{auroc_low:.4f}, {auroc_high:.4f}]")
print(f"AUPRC mean: {auprc_mean:.4f}, 95% CI: [{auprc_low:.4f}, {auprc_high:.4f}]")

## 18. Deterministic explanation workflow

In [ ]:
def generate_risk_summary_from_probability(prob):
    if prob >= 0.70:
        risk_tier = "High Risk"
    elif prob >= 0.30:
        risk_tier = "Intermediate Risk"
    else:
        risk_tier = "Low Risk"

    summary = (
        f"Predicted MI probability: {prob:.3f}. "
        f"Risk tier: {risk_tier}. "
        "This ECG-only model does not use troponin biomarkers, symptoms, "
        "patient history, or clinical examination findings. "
        "This output is intended only for research-based decision support "
        "and is not a standalone diagnosis. Formal clinical evaluation is required."
    )

    return risk_tier, summary

def generate_risk_summary_from_logit(logit):
    prob = 1.0 / (1.0 + np.exp(-float(logit)))
    return generate_risk_summary_from_probability(prob)

for i in range(min(5, len(test_probs))):
    risk_tier, summary = generate_risk_summary_from_probability(test_probs[i])
    print("Example:", i)
    print("True label:", int(test_labels[i]))
    print(summary)
    print("-" * 80)

## 19. Save example risk summaries

In [ ]:
summary_rows = []

for i in range(min(50, len(test_probs))):
    risk_tier, summary = generate_risk_summary_from_probability(test_probs[i])
    summary_rows.append({
        "index": i,
        "true_label": int(test_labels[i]),
        "logit": float(test_logits[i]),
        "probability": float(test_probs[i]),
        "risk_tier": risk_tier,
        "summary": summary,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "example_risk_summaries.csv", index=False)
summary_df.head()

## 20. Paper consistency checklist

After running the notebook, confirm these items before submitting the paper:

1. The data split is:
   - Train: folds 1–8
   - Validation: fold 9
   - Test: fold 10
2. The input tensor shape is `[B, 12, 1000]`.
3. Normalization statistics were fitted only on the training set.
4. The model outputs raw logits.
5. `BCEWithLogitsLoss` receives raw logits, not sigmoid probabilities.
6. The best validation checkpoint is loaded before Fold-10 testing.
7. AUROC and AUPRC in the paper match the printed notebook values.
8. The explanation workflow uses thresholds 0.30 and 0.70.
9. The paper does not claim standalone diagnosis or clinical deployment.
10. The output directory contains:
    - `training_history.csv`
    - `test_results.csv`
    - `roc_curve.png`
    - `precision_recall_curve.png`
    - `example_risk_summaries.csv`